# MIL CNN Experiment Notebook

In [1]:
# %% ─────────────────────────────────────────────────────────────────────────────
!pip install -q torch torchvision torchaudio tqdm pillow matplotlib


Imports & Configs

In [22]:
# PyTorch & friends
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from torchvision import transforms
import torch.nn.functional as F

# Misc
from pathlib import Path
from PIL import Image, ImageOps
import random, math, json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
import numpy as np

from collections import defaultdict
from itertools import chain

import os

In [5]:
# %% ────────────────────────────────────────────────────────────────────────────
# Feel free to tune these
CFG = {
    "data_dir"      : "ASC_dataset_v3_augmented",  # Path to dataset
    "num_classes"   : 2,
    "patch_size"    : 224,             # PxP crops fed to ResNet
    "stride"        : 224,             # Stride for patch extraction
    "batch_size"    : 4,
    "num_workers"   : 0,
    "rand_seed"     : 42,
    "total_epochs"        : 7,
    "fine_tune_epochs"    : 3,
    "lr_backbone_stage1"  : 1e-5,
    "lr_head_stage1"      : 5e-4,
    "lr_head_stage2"      : 5e-5,
    "weight_decay"  : 0.04,
    "device"        : "cuda:1" if torch.cuda.is_available() else "cpu",
    "results_dir": "patch_mil_net_results",
}

random.seed(CFG["rand_seed"])
torch.manual_seed(CFG["rand_seed"])

os.makedirs(CFG["results_dir"], exist_ok=True)

label2id = {"french": 0, "german": 1}

id2label = {v: k for k, v in label2id.items()}


Dataset

In [6]:

def build_bags(split_dir: Path, script_labels=("french", "german")):
    """
    Filenames: {ms_id}_{page_id}_{region_id}_{variant}.ext
    Example: 008_000_00_inv_masked_bin.png

    Returns a list of dicts, each:
        {
            "image_paths": [...],
            "label": int,
            "page_key": "msid_pageid",
            "script_label": "french" / "german"
        }
    """
    bags_dict = defaultdict(lambda: {
        "image_paths": [],
        "label": None,
        "page_key": None,
        "script_label": None,
    })

    for script_label in script_labels:
        class_dir = split_dir / script_label
        if not class_dir.exists():
            continue

        for fname in os.listdir(class_dir):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
                continue

            fpath = class_dir / fname
            stem = os.path.splitext(fname)[0]
            parts = stem.split("_")

            if len(parts) < 3:
                raise ValueError(f"Unexpected filename format: {fname}")

            ms_id   = parts[0]
            page_id = parts[1]
            # region_id = parts[2]
            page_key = f"{ms_id}_{page_id}"
            bag_key  = f"{script_label}_{page_key}"

            bag = bags_dict[bag_key]
            bag["image_paths"].append(str(fpath))
            bag["label"]        = label2id[script_label]
            bag["page_key"]     = page_key
            bag["script_label"] = script_label

    return list(bags_dict.values())


In [7]:


def make_image_transform(train: bool = True):
    """
    Light augmentations on the FULL region image.
    Does NOT resize the full image. Patchification handles the size.
    """
    tfs = []

    if train:
        # Small rotation (very mild to avoid distorting Hebrew letters)
        tfs.append(
            transforms.RandomApply([
                transforms.RandomRotation(
                    degrees=2,
                    fill=0
                )
            ], p=0.5)
        )

        # Small affine jitter
        tfs.append(
            transforms.RandomApply([
                transforms.RandomAffine(
                    degrees=0,
                    translate=(0.02, 0.02),
                    scale=(0.95, 1.05),
                    shear=0,
                    fill=0
                )
            ], p=0.5)
        )

        # Light photometric jitter
        tfs.append(
            transforms.RandomApply([
                transforms.ColorJitter(
                    brightness=0.1,
                    contrast=0.1,
                )
            ], p=0.5)
        )

        # Light blur
        tfs.append(
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
            ], p=0.3)
        )

    # IMPORTANT: do NOT resize image here
    return transforms.Compose(tfs)


def make_patch_transform():
    """
    Transform applied per 224x224 patch.
    Converts to tensor and normalizes for ResNet / ImageNet.
    """
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

In [8]:


class PatchifyBagDataset(Dataset):
    """
    Bag-level dataset with *online* patchification.

    Each item:
        x_bag: [P, 3, patch_size, patch_size]
        y    : scalar label (long)
    """
    def __init__(
        self,
        bags,
        image_transform,
        patch_transform,
        patch_size: int = 224,
        stride: int | None = None,
        max_patches: int | None = None,
        seed: int = 42,
    ):
        """
        Parameters
        ----------
        bags : list[dict]
            Output of build_bags(). Each dict:
              - "image_paths": list[str]
              - "label": int
        image_transform : callable
            Applied to full region image before patchifying.
        patch_transform : callable
            Applied to each cropped patch.
        patch_size : int
            Patch size in pixels (patch_size x patch_size).
        stride : int or None
            Stride for the patch grid. If None, uses stride = patch_size
            (non-overlapping grid). You can set smaller stride for overlap.
        max_patches : int or None
            If set, fixes number of patches per bag:
              - if > produced patches: oversample with replacement
              - if < produced patches: subsample randomly
        seed : int
            Random seed for sampling.
        """
        self.bags = bags
        self.image_transform = image_transform
        self.patch_transform = patch_transform
        self.patch_size = patch_size
        self.stride = stride if stride is not None else patch_size
        self.max_patches = max_patches
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.bags)

    def _patchify_image(self, img: Image.Image):
        """
        Patchify a PIL image into a list of [3, patch_size, patch_size] tensors.
        """
        w, h = img.size
        ps = self.patch_size
        st = self.stride

        patches = []
        if w < ps or h < ps:
            # Fallback: resize to exact patch_size and take single patch
            img_resized = img.resize((ps, ps))
            patches.append(self.patch_transform(img_resized))
            return patches

        # Regular grid (no “tail” handling; simple and consistent)
        for top in range(0, h - ps + 1, st):
            for left in range(0, w - ps + 1, st):
                crop = img.crop((left, top, left + ps, top + ps))
                patches.append(self.patch_transform(crop))

        return patches

    def _sample_patches(self, patches):
        if self.max_patches is None:
            return patches

        n = len(patches)
        if n == 0:
            raise RuntimeError("No patches produced for a bag; check image sizes / transforms.")

        if n > self.max_patches:
            idxs = self.rng.sample(range(n), self.max_patches)
            return [patches[i] for i in idxs]
        elif n < self.max_patches:
            extra = [self.rng.choice(patches) for _ in range(self.max_patches - n)]
            return patches + extra
        else:
            return patches

    def __getitem__(self, idx):
        bag_info = self.bags[idx]
        image_paths = bag_info["image_paths"]
        label = int(bag_info["label"])

        all_patches = []

        for path in image_paths:
            img = Image.open(path).convert("RGB")
            img = self.image_transform(img)
            patches = self._patchify_image(img)
            all_patches.extend(patches)

        if len(all_patches) == 0:
            raise RuntimeError(f"No patches produced for bag index {idx} (image_paths={image_paths}).")

        all_patches = self._sample_patches(all_patches)

        x_bag = torch.stack(all_patches, dim=0)  # [P, 3, patch_size, patch_size]
        y = torch.tensor(label, dtype=torch.long)

        return x_bag, y


Model

In [9]:

def make_resnet_backbone(name: str = "resnet18", pretrained: bool = True, cut_last: int = 2):
    """
    Build a torchvision ResNet backbone and cut off the last layers.

    Parameters
    ----------
    name : {"resnet18", "resnet34", "resnet50"}
    pretrained : bool
        If True, use ImageNet weights.
    cut_last : int
        Number of trailing modules to drop. With cut_last=2, we drop
        avgpool and fc, keeping conv1..layer4.

    Returns
    -------
    backbone : nn.Sequential
        Feature extractor (conv1..layer4).
    out_dim : int
        Feature dimension after global average pooling.
    """
    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        base = models.resnet18(weights=weights)
    elif name == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        base = models.resnet34(weights=weights)
    elif name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        base = models.resnet50(weights=weights)
    else:
        raise ValueError(f"Unsupported backbone: {name}")

    # children: conv1, bn1, relu, maxpool, layer1–4, avgpool, fc
    backbone = nn.Sequential(*list(base.children())[:-cut_last])
    out_dim = base.fc.in_features  # 512 for resnet18/34, 2048 for resnet50
    return backbone, out_dim


In [10]:
class PatchMILNet(nn.Module):
    """
    MIL network with configurable ResNet backbone.

    backbone_name ∈ {"resnet18", "resnet34", "resnet50"}.
    For your “smaller encoder” baseline, use "resnet18".
    """
    def __init__(
        self,
        num_classes: int,
        backbone_name: str = "resnet18",
        hidden: int = 256,
        dropout: float = 0.3,
        pretrained: bool = True,
    ):
        super().__init__()

        # Backbone: conv1..layer4 (no avgpool, no fc)
        self.backbone, self.backbone_dim = make_resnet_backbone(
            name=backbone_name,
            pretrained=pretrained,
            cut_last=2,
        )

        # Global pooling to get one feature vector per patch
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Dropout on pooled features
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Patch classifier head
        self.cls = nn.Linear(self.backbone_dim, num_classes)

        # Attention head
        self.att = nn.Sequential(
            nn.Linear(self.backbone_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor):
        """
        x: [B, P, 3, H, W]
        """
        B, P, C, H, W = x.shape
        x = x.view(B * P, C, H, W)          # [B*P, 3, H, W]

        feats = self.backbone(x)            # [B*P, D, h, w]
        feats = self.global_pool(feats).squeeze(-1).squeeze(-1)  # [B*P, D]
        feats = self.dropout(feats)

        logits_patch = self.cls(feats)      # [B*P, C]
        att_scores   = self.att(feats)      # [B*P, 1]

        logits_patch = logits_patch.view(B, P, -1)  # [B, P, C]
        att_scores   = att_scores.view(B, P)        # [B, P]

        att_w = torch.softmax(att_scores, dim=1).unsqueeze(-1)   # [B, P, 1]
        logits_page = (att_w * logits_patch).sum(dim=1)          # [B, C]

        return logits_page, logits_patch, att_w


Train!

In [11]:


train_bags = build_bags(Path(CFG["data_dir"]) / "train")
val_bags   = build_bags(Path(CFG["data_dir"]) / "val")

patch_size = CFG["patch_size"]
stride     = CFG["stride"]

train_img_tf = make_image_transform(train=True)
val_img_tf   = make_image_transform(train=False)
patch_tf     = make_patch_transform()

train_ds = PatchifyBagDataset(
    train_bags,
    image_transform=train_img_tf,
    patch_transform=patch_tf,
    patch_size=patch_size,
    stride=stride,
    max_patches=64,
)

val_ds = PatchifyBagDataset(
    val_bags,
    image_transform=val_img_tf,
    patch_transform=patch_tf,
    patch_size=patch_size,
    stride=stride,
    max_patches=400,
)

train_dl = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"])
val_dl   = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"])

In [ ]:


def set_backbone_requires_grad(model: PatchMILNetResNet50, requires_grad: bool):
    for p in model.backbone.parameters():
        p.requires_grad = requires_grad


def make_optimizer_stage1(model, lr_backbone: float, lr_head: float, weight_decay: float):
    """
    Stage 1: backbone unfrozen, but with smaller LR than head.
    """
    # Different LR for backbone vs head
    params = [
        {"params": model.backbone.parameters(), "lr": lr_backbone},
        {"params": chain(model.cls.parameters(), model.att.parameters()), "lr": lr_head},
    ]
    return torch.optim.AdamW(params, weight_decay=weight_decay)


def make_optimizer_stage2(model, lr_head: float, weight_decay: float):
    """
    Stage 2: backbone frozen, optimiser only over the head.
    """
    head_params = chain(model.cls.parameters(), model.att.parameters())
    return torch.optim.AdamW(head_params, lr=lr_head, weight_decay=weight_decay)


In [11]:

def train_epoch(model, loader, crit, opt, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader)

    for bags, labels in pbar:        # bags: [B, P, 3, H, W]
        bags   = bags.to(device)
        labels = labels.to(device)

        opt.zero_grad()

        logits_page, _, _ = model(bags)        # [B, C]
        loss = crit(logits_page, labels)

        loss.backward()
        opt.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        preds = logits_page.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += batch_size

        pbar.set_description(
            f"train loss {total_loss/total:.4f} | acc {correct/total:.3f}"
        )

    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, crit, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for bags, labels in loader:
        bags   = bags.to(device)
        labels = labels.to(device)

        logits_page, _, _ = model(bags)
        loss = crit(logits_page, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        preds = logits_page.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += batch_size

    return total_loss / total, correct / total


In [ ]:

device = CFG["device"]

# Hyperparameters for 2 stages
FINE_TUNE_EPOCHS    = CFG.get("fine_tune_epochs", 3) 
TOTAL_EPOCHS        = CFG["total_epochs"]
LR_BACKBONE_STAGE1  = CFG["lr_backbone_stage1"]
LR_HEAD_STAGE1      = CFG["lr_head_stage1"]
LR_HEAD_STAGE2      = CFG["lr_head_stage2"]
WEIGHT_DECAY        = CFG["weight_decay"]


model = PatchMILNet(
    num_classes=CFG["num_classes"],
    backbone_name="resnet34",
    hidden=256,
    dropout=0.3,
    pretrained=True,
).to(device)

criterion = nn.CrossEntropyLoss()

best_val_acc  = 0.0
best_val_loss = float("inf")
best_ckpt_path = CFG["results_dir"] + "/best_mil_patchnet_2stage.pth"

# -----------------------------
# Stage 1: backbone UNFROZEN
# -----------------------------
print("Stage 1: fine-tuning backbone (unfrozen)")

set_backbone_requires_grad(model, True)
optimizer = make_optimizer_stage1(
    model,
    lr_backbone=LR_BACKBONE_STAGE1,
    lr_head=LR_HEAD_STAGE1,
    weight_decay=WEIGHT_DECAY,
)


# unfreeze_layer4_only(model)

# optimizer = torch.optim.AdamW(
    # [
        # {"params": model.backbone[7].parameters(),               "lr": LR_BACKBONE_STAGE1},
        # {"params": model.cls.parameters(),                       "lr": LR_HEAD_STAGE1},
        # {"params": model.att.parameters(),                       "lr": LR_HEAD_STAGE1},
    # ],
    # weight_decay=WEIGHT_DECAY,
# )

for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_dl, criterion, optimizer, device)
    val_loss, val_acc = eval_epoch(model, val_dl, criterion, device)

    print(f"[Stage 1] Epoch {epoch:02d} | train loss {tr_loss:.4f} | train acc {tr_acc:.3f}")
    print(f"[Stage 1] Epoch {epoch:02d} |   val loss {val_loss:.4f} |   val acc {val_acc:.3f}")

    is_better = (val_acc > best_val_acc) or (
        (val_acc == best_val_acc) and (val_loss < best_val_loss)
    )

    if is_better:
        best_val_acc  = val_acc
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt_path)
        print(f"  -> New best model saved (Stage 1): acc={best_val_acc:.3f}, loss={best_val_loss:.4f}")


# -----------------------------
# Stage 2: backbone FROZEN
# -----------------------------
print("Stage 2: freezing backbone, training MIL head only")

set_backbone_requires_grad(model, False)
optimizer = make_optimizer_stage2(
    model,
    lr_head=LR_HEAD_STAGE2,
    weight_decay=WEIGHT_DECAY,
)


# freeze_all_backbone(model)

# optimizer = torch.optim.AdamW(
    # [
        # {"params": model.cls.parameters(), "lr": LR_HEAD_STAGE2},
        # {"params": model.att.parameters(), "lr": LR_HEAD_STAGE2},
    # ],
    # weight_decay=WEIGHT_DECAY,
# )

for epoch in range(FINE_TUNE_EPOCHS + 1, TOTAL_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_dl, criterion, optimizer, device)
    val_loss, val_acc = eval_epoch(model, val_dl, criterion, device)

    print(f"[Stage 2] Epoch {epoch:02d} | train loss {tr_loss:.4f} | train acc {tr_acc:.3f}")
    print(f"[Stage 2] Epoch {epoch:02d} |   val loss {val_loss:.4f} |   val acc {val_acc:.3f}")

    is_better = (val_acc > best_val_acc) or (
        (val_acc == best_val_acc) and (val_loss < best_val_loss)
    )

    if is_better:
        best_val_acc  = val_acc
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt_path)
        print(f"  -> New best model saved (Stage 2): acc={best_val_acc:.3f}, loss={best_val_loss:.4f}")


Stage 1: fine-tuning backbone (unfrozen)


train loss 0.7286 | acc 0.525: 100%|██████████| 26/26 [03:04<00:00,  7.09s/it]


[Stage 1] Epoch 01 | train loss 0.7286 | train acc 0.525
[Stage 1] Epoch 01 |   val loss 0.6326 |   val acc 0.625
  -> New best model saved (Stage 1): acc=0.625, loss=0.6326


train loss 0.7057 | acc 0.554: 100%|██████████| 26/26 [03:00<00:00,  6.93s/it]


[Stage 1] Epoch 02 | train loss 0.7057 | train acc 0.554
[Stage 1] Epoch 02 |   val loss 0.6268 |   val acc 0.625
  -> New best model saved (Stage 1): acc=0.625, loss=0.6268


train loss 0.5154 | acc 0.782: 100%|██████████| 26/26 [03:01<00:00,  6.99s/it]


... (5 outputs left out)

train loss 0.3242 | acc 0.881: 100%|██████████| 26/26 [02:55<00:00,  6.77s/it]


[Stage 2] Epoch 06 | train loss 0.3242 | train acc 0.881
[Stage 2] Epoch 06 |   val loss 0.3294 |   val acc 0.938


train loss 0.3451 | acc 0.891: 100%|██████████| 26/26 [02:56<00:00,  6.80s/it]


[Stage 2] Epoch 07 | train loss 0.3451 | train acc 0.891
[Stage 2] Epoch 07 |   val loss 0.3546 |   val acc 0.812


Inference

In [18]:

patch_size = CFG["patch_size"]
stride     = CFG["stride"]

test_split_dir = Path(CFG["data_dir"]) / "test"

# Build bags for test split
test_bags = build_bags(test_split_dir)

# Transforms: no training augmentations, just deterministic
test_img_tf = make_image_transform(train=False)
patch_tf    = make_patch_transform()

# Full-bag dataset: max_patches=None → use ALL patches from all regions/variants
test_ds = PatchifyBagDataset(
    test_bags,
    image_transform=test_img_tf,
    patch_transform=patch_tf,
    patch_size=patch_size,
    stride=stride,
    max_patches=None,   # FULL BAG
)

test_dl = DataLoader(
    test_ds,
    batch_size=1,       # easiest to align bags↔predictions
    shuffle=False,
    num_workers=CFG["num_workers"],
)

val_ds = PatchifyBagDataset(
    val_bags,
    image_transform=val_img_tf,
    patch_transform=patch_tf,
    patch_size=patch_size,
    stride=stride,
)

val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CFG["num_workers"])

In [19]:

device = CFG["device"]

checkpoint_path = CFG["results_dir"] + "/best_mil_patchnet_2stage.pth"

# Recreate the model architecture exactly as in training
model = PatchMILNet(
    num_classes=CFG["num_classes"],
    backbone_name="resnet34",
    hidden=256,
    dropout=0.3,
    pretrained=False,
).to(device)
state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state)
model.eval()


PatchMILNet(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

In [23]:

@torch.no_grad()
def infer_on_ds(model, loader, bags, results_dir, split: str, device, id2label):
    """
    Saves a CSV with columns:
      page_key, p(French), p(German), pred_label, true_label, num_patches
    """
    model.eval()
    records = []

    # Assumes loader iterates in the same order as `bags`
    bag_ptr = 0

    for xb, _ in loader:
        xb = xb.to(device)  # xb: [B, P, 3, ps, ps]
        B, P = xb.shape[0], xb.shape[1]

        logits_page, _, _ = model(xb)          # [B, C]
        probs = F.softmax(logits_page, dim=1)  # [B, C]
        preds = probs.argmax(dim=1)            # [B]

        for b in range(B):
            bag_info = bags[bag_ptr]
            bag_ptr += 1

            page_key = bag_info.get("page_key", f"bag_{bag_ptr-1}")
            true_id = int(bag_info["label"])
            true_label = id2label[true_id]

            pred_id = int(preds[b].item())
            pred_label = id2label[pred_id]

            # Map probabilities to French/German explicitly (order-safe)
            # This assumes your id2label uses keys like "french"/"german" (case-insensitive handled below).
            label_names = {v.lower(): k for k, v in id2label.items()}  # e.g. {"french":0, "german":1}
            french_id = label_names.get("french", None)
            german_id = label_names.get("german", None)
            if french_id is None or german_id is None:
                raise ValueError("id2label must contain 'french' and 'german' labels.")

            p_french = float(probs[b, french_id].item())
            p_german = float(probs[b, german_id].item())

            records.append(
                {
                    "page_key": page_key,
                    "p(French)": p_french,
                    "p(German)": p_german,
                    "pred_label": pred_label,
                    "true_label": true_label,
                    "num_patches": int(P),
                }
            )

    os.makedirs(results_dir, exist_ok=True)
    out_path = os.path.join(results_dir, f"patchmilnet_{split}_predictions.csv")
    df = pd.DataFrame(records)

    # Optional: stable, thesis-friendly ordering
    df = df.sort_values("page_key").reset_index(drop=True)

    df.to_csv(out_path, index=False)
    print(f"Saved predictions to: {out_path}")
    return df


# Usage
results_dir = CFG["results_dir"]
df_test_results = infer_on_ds(model, test_dl, test_bags, results_dir, split="test", device=CFG["device"], id2label=id2label)
df_val_results  = infer_on_ds(model, val_dl,  val_bags,  results_dir, split="val",  device=CFG["device"], id2label=id2label)


Saved predictions to: patch_mil_net_results/patchmilnet_test_predictions.csv
Saved predictions to: patch_mil_net_results/patchmilnet_val_predictions.csv


Visualize Attention Maps

In [24]:

def compute_attention_for_bag(
    model,
    bag_info,
    patch_size: int = 224,
    stride: int = 224,
    patch_transform=None,
    device: str = "cuda",
):
    """
    Compute MIL attention weights per patch and organize them into
    patch grids per region image.

    Returns
    -------
    per_image_att : list of dicts
        Each dict contains:
            - "image_path": str
            - "att_grid": np.ndarray of shape [n_rows, n_cols]
            - "H": original image height
            - "W": original image width
    """
    model.eval()

    if patch_transform is None:
        patch_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    patches = []
    meta = []  # (img_idx, top, left, H, W)

    image_paths = bag_info["image_paths"]

    # 1) Patchify all region images
    for img_idx, path in enumerate(image_paths):
        img = Image.open(path).convert("RGB")
        W, H = img.size

        # simple regular grid; must match training patchify logic
        if W < patch_size or H < patch_size:
            # fallback: single resized patch
            crop = img.resize((patch_size, patch_size))
            patches.append(patch_transform(crop))
            meta.append((img_idx, 0, 0, patch_size, patch_size))
        else:
            for top in range(0, H - patch_size + 1, stride):
                for left in range(0, W - patch_size + 1, stride):
                    crop = img.crop((left, top, left + patch_size, top + patch_size))
                    patches.append(patch_transform(crop))
                    meta.append((img_idx, top, left, H, W))

    if len(patches) == 0:
        raise RuntimeError(f"No patches produced for bag with page_key={bag_info.get('page_key')}")

    # 2) Run through model to get attention weights
    x = torch.stack(patches, dim=0).unsqueeze(0).to(device)  # [1, P, 3, ps, ps]
    with torch.no_grad():
        _, _, att_w = model(x)       # att_w: [1, P, 1]

    att = att_w.squeeze(0).squeeze(-1).cpu().numpy()  # [P]

    # 3) Build per-image attention grids
    # We first collect patch indices per image
    img_to_patches = {}
    for i, (img_idx, top, left, H, W) in enumerate(meta):
        if img_idx not in img_to_patches:
            img_to_patches[img_idx] = {
                "coords": [],
                "H": H,
                "W": W,
            }
        img_to_patches[img_idx]["coords"].append((top, left, i))

    per_image_att = []

    for img_idx, info in img_to_patches.items():
        coords = info["coords"]
        H, W = info["H"], info["W"]

        # infer grid shape from max top/left
        if H < patch_size or W < patch_size:
            n_rows, n_cols = 1, 1
        else:
            n_rows = (H - patch_size) // stride + 1
            n_cols = (W - patch_size) // stride + 1

        grid = np.zeros((n_rows, n_cols), dtype=np.float32)

        for (top, left, patch_i) in coords:
            if H < patch_size or W < patch_size:
                r, c = 0, 0
            else:
                r = top // stride
                c = left // stride
            grid[r, c] = att[patch_i]

        per_image_att.append({
            "image_path": image_paths[img_idx],
            "att_grid": grid,
            "H": H,
            "W": W,
        })

    return per_image_att


In [28]:

def save_attention_overlay(
    image_path,
    att_grid,
    H,
    W,
    out_path,
    cmap_name="magma",     # better than "jet" for scientific figures
    alpha=0.30,            # slightly more subtle
    clip_percentiles=(5, 95),  # robust normalization; set None to disable
    add_colorbar=False,    # usually OFF in thesis figures
    save_original=False,   # optionally save original next to overlay
):
    """
    Create and save an overlay of attention heatmap on the original image.

    att_grid: [n_rows, n_cols] attention values (one per patch)
    H, W: original image height/width
    """

    img = Image.open(image_path).convert("RGB")

    # -------------------------
    # Normalize attention grid
    # -------------------------
    att = att_grid.astype(np.float32)

    # Robust scaling so one hot patch doesn't dominate
    if clip_percentiles is not None:
        lo, hi = np.percentile(att, clip_percentiles)
        if hi > lo:
            att = np.clip(att, lo, hi)

    att_min, att_max = att.min(), att.max()
    if att_max > att_min:
        att_norm = (att - att_min) / (att_max - att_min)
    else:
        att_norm = np.zeros_like(att, dtype=np.float32)

    # -------------------------
    # Upsample to full image
    # -------------------------
    att_img = Image.fromarray((att_norm * 255).astype(np.uint8))
    att_img = att_img.resize((W, H), resample=Image.BILINEAR)
    att_norm_up = np.array(att_img, dtype=np.float32) / 255.0  # [H, W]

    # -------------------------
    # Colormap + blending
    # -------------------------
    cmap = cm.get_cmap(cmap_name)
    att_color = cmap(att_norm_up)[..., :3].astype(np.float32)  # [H, W, 3]

    img_np = np.array(img).astype(np.float32) / 255.0
    overlay = (1.0 - alpha) * img_np + alpha * att_color
    overlay = np.clip(overlay, 0.0, 1.0)

    # -------------------------
    # Save outputs
    # -------------------------
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Optionally save original image for a paired figure
    if save_original:
        orig_path = os.path.splitext(out_path)[0] + "_original.png"
        img.save(orig_path)

    plt.figure(figsize=(6, 6))
    plt.imshow(overlay)
    plt.axis("off")

    if add_colorbar:
        # Create a mappable for the colorbar
        mappable = cm.ScalarMappable(cmap=cmap)
        mappable.set_array(att_norm_up)
        plt.colorbar(mappable, fraction=0.046, pad=0.04)

    plt.tight_layout(pad=0)
    plt.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close()


In [29]:
device = CFG["device"]
patch_size = CFG["patch_size"]
stride     = CFG.get("stride", patch_size)
results_dir = CFG["results_dir"]

# Recreate model and load best checkpoint
model = PatchMILNet(
    num_classes=CFG["num_classes"],
    backbone_name="resnet34",
    hidden=256,
    dropout=0.3,
    pretrained=False,
).to(device)
state = torch.load(results_dir + "/best_mil_patchnet_2stage.pth", map_location=device)
model.load_state_dict(state)
model.eval()

page_keys = {"009_000", "021_000", "025_000"}  # use a set for O(1) lookup

bags_to_visualize = [
    b for b in test_bags
    if b.get("page_key") in page_keys
]

for i, bag_info in enumerate(bags_to_visualize):
    page_key = bag_info.get("page_key", f"bag_{i}")

    per_image_att = compute_attention_for_bag(
        model,
        bag_info,
        patch_size=patch_size,
        stride=stride,
        patch_transform=make_patch_transform(),  # same normalization as training
        device=device,
    )

    vis_dir = os.path.join(results_dir, "attention_vis", page_key)

    for j, info in enumerate(per_image_att):
        img_path = info["image_path"]
        att_grid = info["att_grid"]
        H, W     = info["H"], info["W"]

        # name result by region index or by filename stem
        stem = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(vis_dir, f"{stem}_att_overlay.png")

        save_attention_overlay(
            image_path=img_path,
            att_grid=att_grid,
            H=H, W=W,
            out_path=out_path,
            cmap_name="magma",
            alpha=0.30,
            clip_percentiles=(5, 95),
            add_colorbar=False,
            save_original=True,   # so you can show original + overlay side-by-side
        )

    print(f"Saved attention overlay: {out_path}")


/tmp/ipykernel_80714/1198060627.py:49: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(cmap_name)


Saved attention overlay: patch_mil_net_results/attention_vis/009_000/009_000_00_att_overlay.png
Saved attention overlay: patch_mil_net_results/attention_vis/021_000/021_000_00_att_overlay.png
Saved attention overlay: patch_mil_net_results/attention_vis/025_000/025_000_00_att_overlay.png
